In [ ]:
import pandas as pd
from pathlib import Path

# Set your Parquet file path here
parquet_path = Path("ohlcv_data.parquet")

df = pd.read_parquet(parquet_path)
df

,symbol,date,open,high,low,close,volume
119,BRK-B,2016-01-08,130.110001,130.399994,128.210007,128.330002,6101600
120,BRK-B,2016-01-13,128.970001,129.380005,125.709999,126.250000,6042400
121,BRK-B,2016-01-15,125.339996,126.809998,124.510002,126.139999,8145100
122,BRK-B,2016-01-29,126.660004,129.770004,126.110001,129.770004,6523800
123,BRK-B,2016-02-03,126.339996,126.629997,123.550003,126.239998,6607500
...,...,...,...,...,...,...,...
4365393,VICI,2020-07-02,15.681162,15.740225,15.260340,15.400615,4136400
4365394,VICI,2020-07-06,15.799288,15.865734,14.902272,14.928112,4423500
4365395,VICI,2020-07-07,14.691864,15.105303,14.573738,14.758309,2779800
4365396,VICI,2020-07-08,14.809990,14.950265,14.621728,14.913351,2158800


In [12]:
# Filter the data from January 1, 2016 to December 31, 2025
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df[(df['date'] >= '2016-01-01') & (df['date'] <= '2025-12-31')].copy()

# Filter to the first 500 unique stocks in the existing file order
first_500_symbols = df['symbol'].dropna().astype(str).drop_duplicates().head(500)
df = df[df['symbol'].astype(str).isin(first_500_symbols)].copy()

In [13]:
df.info()

<class 'pandas.DataFrame'>
Index: 1220278 entries, 119 to 4365397
Data columns (total 7 columns):
 #   Column  Non-Null Count    Dtype        
---  ------  --------------    -----        
 0   symbol  1220278 non-null  str          
 1   date    1220278 non-null  datetime64[s]
 2   open    1220278 non-null  float64      
 3   high    1220278 non-null  float64      
 4   low     1220278 non-null  float64      
 5   close   1220278 non-null  float64      
 6   volume  1220278 non-null  int64        
dtypes: datetime64[s](1), float64(4), int64(1), str(1)
memory usage: 78.3 MB


In [14]:
df.isna().sum()

symbol    0
date      0
open      0
high      0
low       0
close     0
volume    0
dtype: int64

In [16]:
df['Adj_Close'] = df['close']  # yfinance adjusted by default
df['Daily_Return'] = df.groupby('symbol')['close'].pct_change()
df['Target_Signal'] = (df['Daily_Return'] > 0).astype(int)

# Build a sector lookup when one is not already available.
try:
    import yfinance as yf
except ImportError:
    yf = None

sector_dict = globals().get('sector_dict', {}) or {}
if not sector_dict and yf is not None:
    sector_dict = {}
    for symbol in df['symbol'].dropna().astype(str).drop_duplicates():
        try:
            info = yf.Ticker(symbol).info
            sector_dict[symbol] = info.get('sector') or 'Unknown'
        except Exception:
            sector_dict[symbol] = 'Unknown'

df['Sector'] = df['symbol'].map(sector_dict).fillna('Unknown')

c:\Users\Admin\anaconda3\Lib\site-packages\yfinance\scrapers\quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
c:\Users\Admin\anaconda3\Lib\site-packages\yfinance\scrapers\quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")


In [19]:
df

,symbol,date,open,high,low,close,volume,Adj_Close,Daily_Return,Target_Signal,Sector
119,BRK-B,2016-01-08,130.110001,130.399994,128.210007,128.330002,6101600,128.330002,NaN,0,Financial Services
120,BRK-B,2016-01-13,128.970001,129.380005,125.709999,126.250000,6042400,126.250000,-0.016208,0,Financial Services
121,BRK-B,2016-01-15,125.339996,126.809998,124.510002,126.139999,8145100,126.139999,-0.000871,0,Financial Services
122,BRK-B,2016-01-29,126.660004,129.770004,126.110001,129.770004,6523800,129.770004,0.028778,1,Financial Services
123,BRK-B,2016-02-03,126.339996,126.629997,123.550003,126.239998,6607500,126.239998,-0.027202,0,Financial Services
...,...,...,...,...,...,...,...,...,...,...,...
4365393,VICI,2020-07-02,15.681162,15.740225,15.260340,15.400615,4136400,15.400615,0.007729,1,Real Estate
4365394,VICI,2020-07-06,15.799288,15.865734,14.902272,14.928112,4423500,14.928112,-0.030681,0,Real Estate
4365395,VICI,2020-07-07,14.691864,15.105303,14.573738,14.758309,2779800,14.758309,-0.011375,0,Real Estate
4365396,VICI,2020-07-08,14.809990,14.950265,14.621728,14.913351,2158800,14.913351,0.010505,1,Real Estate
